In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_15_try_1.pickle

trying: ['dt', 'orig_output']
me:  18
me:  19
trying: ['dt', 'orig_output']
me:  18
me:  19
trying: ['flights_df']


me:  20
trying: ['i']
me:  21
trying: ['flight']
me:  21
trying: ['carrier']
me:  21
trying: ['dest']
me:  21
trying: ['df']
me:  21
trying: ['count']
me:  21
trying: ['result']
me:  21
trying: ['df2']
me:  10
trying: ['f_total']
me:  8
trying: ['f']
me:  8
trying: ['plt']
me:  0
trying: ['IPython']
me:  0
trying: ['time']
me:  0
trying: ['np']
me:  0
trying: ['sklearn']
me:  0
trying: ['gdf']
me:  22
trying: ['sp']
me:  0
trying: ['max_speed']
me:  20
trying: ['matplotlib']
me:  0
trying: ['speed']
me:  20
trying: ['ds']
me:  16
trying: ['pd']
me:  0
Declaring variable plt
Declaring variable IPython
Declaring variable time
Declaring variable np
Declaring variable sklearn
Declaring variable sp
Declaring variable matplotlib
Declaring variable pd
Declaring variable f_total
Declaring variable f
Declaring variable df2
Declaring variable ds
Declaring variable dt
Declaring variable dt
Declaring variable orig_output
Declaring variable orig_output
Declaring variable flights_df
Declaring variab

In [4]:
%%RecordEvent
%%time
### cell 16 ###

# filter to June
df = flights_df[flights_df['month'] == 6]

# GPU‐native groupby.mean (avoids the CPU fallback of .agg)
df = df.groupby('carrier', as_index=False)[['arr_delay', 'dep_delay']].mean()

# ensure the rows are in the same (alphabetical) order as the original pandas .agg
df = df.sort_values('carrier').reset_index(drop=True)

# vectorized total delay on GPU
df['total_delay'] = df['arr_delay'] + df['dep_delay']

# pick out the minimum in one go, rather than looping
min_idx = df['total_delay'].idxmin()
print(df['carrier'].iloc[min_idx], df['total_delay'].iloc[min_idx])

# return the DataFrame
df

HA 3.3
CPU times: user 35.3 ms, sys: 39.8 ms, total: 75.1 ms
Wall time: 75.1 ms


,carrier,arr_delay,dep_delay,total_delay
0,9E,NaN,NaN,NaN
1,AA,NaN,NaN,NaN
2,AS,-3.750000,13.083333,9.333333
3,B6,NaN,NaN,NaN
4,DL,NaN,NaN,NaN
5,EV,NaN,NaN,NaN
6,F9,26.636364,29.436364,56.072727
7,FL,NaN,NaN,NaN
8,HA,1.833333,1.466667,3.300000
9,MQ,NaN,NaN,NaN


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_16_try_1.pickle

migration speed (bps): 778334384.3827751
---------------------------
variables to migrate:
ds 48
count 28
result 48
orig_output 48
dt 48
f_total 28
f 48
df 48
pd 72
df2 48
max_speed 32
min_idx 28
sklearn 72
sp 72
matplotlib 72
plt 72
IPython 72
time 72
np 72
speed 48
flights_df 48
gdf 48
i 28
flight 28
carrier 51
dest 52
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_16_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['flights_df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 6 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 7 =====

In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/opt_cell_exec_info_16_try_1.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[16], f)

In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/annotated/checkpoints/post_cell_16.pickle
assert compare_df(df_opt, df)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['flights_df', 'gdf']


me:  20
me:  24
trying: ['flights_df', 'gdf']
me:  20
me:  24
trying: ['orig_output', 'df']
me:  27
me:  26
trying: ['orig_output', 'df']


me:  27
me:  26
trying: ['counts']
me:  24
trying: ['complete_year']
me:  24
trying: ['f']
me:  8
trying: ['f_total']
me:  8
trying: ['matplotlib']
me:  0
trying: ['sp']
me:  0
trying: ['IPython']
me:  0
trying: ['count']
me:  22
trying: ['i']
me:  26
trying: ['pd']
me:  0
trying: ['time']
me:  0
trying: ['df2']
me:  10
trying: ['sklearn']
me:  0
trying: ['np']
me:  0
trying: ['ds']
me:  16
trying: ['dt']
me:  18
trying: ['plt']
me:  0


Declaring variable matplotlib
Declaring variable sp
Declaring variable IPython
Declaring variable pd
Declaring variable time
Declaring variable sklearn
Declaring variable np
Declaring variable plt
Declaring variable f
Declaring variable f_total
Declaring variable df2
Declaring variable ds
Declaring variable dt
Declaring variable flights_df
Declaring variable flights_df
Declaring variable count
Declaring variable gdf
Declaring variable gdf
Declaring variable counts
Declaring variable complete_year
Declaring variable df
Declaring variable df
Declaring variable i
Declaring variable orig_output
Declaring variable orig_output


ValueError: Content of df1 and df2 do not match